In [1]:
import requests
import pandas as pd
from datetime import datetime

# Your API key (keep this private - don't share or push to GitHub with the real key visible)
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")

print("API key loaded:", "Yes" if API_KEY else "No key found!")

# Cities to check
cities = ["Lagos", "London", "New York"]

# Test with just one city first
city = "Lagos"
url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

response = requests.get(url)
print("Status Code:", response.status_code)
print(response.json())

API key loaded: Yes
Status Code: 200
{'coord': {'lon': 3.75, 'lat': 6.5833}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 26.22, 'feels_like': 26.22, 'temp_min': 26.22, 'temp_max': 26.22, 'pressure': 1012, 'humidity': 80, 'sea_level': 1012, 'grnd_level': 1012}, 'visibility': 10000, 'wind': {'speed': 2.71, 'deg': 230, 'gust': 4.95}, 'clouds': {'all': 100}, 'dt': 1787060451, 'sys': {'country': 'NG', 'sunrise': 1787031567, 'sunset': 1787075920}, 'timezone': 3600, 'id': 2332453, 'name': 'Lagos', 'cod': 200}


In [2]:
cities = ["Lagos", "London", "New York"]
raw_data = []

for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        raw_data.append(response.json())
        print(f"✓ Got data for {city}")
    else:
        print(f"✗ Failed for {city}: {response.status_code}")

✓ Got data for Lagos
✓ Got data for London
✓ Got data for New York


In [3]:
records = []
for data in raw_data:
    records.append({
        "City": data["name"],
        "Temperature_C": data["main"]["temp"],
        "Humidity_%": data["main"]["humidity"],
        "Weather_Condition": data["weather"][0]["description"],
        "Wind_Speed_ms": data["wind"]["speed"],
        "DateTime": datetime.fromtimestamp(data["dt"])
    })

df = pd.DataFrame(records)
df

,City,Temperature_C,Humidity_%,Weather_Condition,Wind_Speed_ms,DateTime
0,Lagos,26.22,80,overcast clouds,2.71,2026-08-18 14:40:51
1,London,25.29,57,overcast clouds,4.63,2026-08-18 14:35:27
2,New York,23.89,91,broken clouds,2.24,2026-08-18 14:39:39


In [4]:
# Save as CSV
df.to_csv("weather_data.csv", index=False)

# Save to SQLite
import sqlite3
conn = sqlite3.connect("weather_data.db")
df.to_sql("weather", conn, if_exists="replace", index=False)
conn.close()

print("Data saved to CSV and SQLite database")

Data saved to CSV and SQLite database


In [5]:
print("Temperature comparison:")
print(df[["City", "Temperature_C"]].sort_values("Temperature_C", ascending=False))

print("\nHighest humidity city:")
print(df.loc[df["Humidity_%"].idxmax(), ["City", "Humidity_%"]])

print("\nWeather conditions:")
print(df[["City", "Weather_Condition"]])

Temperature comparison:
       City  Temperature_C
0     Lagos          26.22
1    London          25.29
2  New York          23.89

Highest humidity city:
City          New York
Humidity_%          91
Name: 2, dtype: object

Weather conditions:
       City Weather_Condition
0     Lagos   overcast clouds
1    London   overcast clouds
2  New York     broken clouds
